# SupplyMind AI — XGBoost

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
# -------------------
# Reload evaluation code
# -------------------

import importlib

import supplymind.features.predictions.ml.evaluation as evaluation

evaluation = importlib.reload(evaluation)

positive_class_probability = evaluation.positive_class_probability
choose_threshold = evaluation.choose_threshold
evaluate_probabilities = evaluation.evaluate_probabilities
BinaryMetrics = evaluation.BinaryMetrics

print("Evaluation module:", evaluation.__file__)
print("BinaryMetrics fields:", BinaryMetrics.__annotations__)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_xgboost,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

Evaluation module: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/src/supplymind/features/predictions/ml/evaluation.py
BinaryMetrics fields: {'accuracy': 'float', 'precision': 'float', 'recall': 'float', 'f1': 'float', 'roc_auc': 'float', 'average_precision': 'float', 'balanced_accuracy': 'float', 'specificity': 'float', 'true_negative': 'int', 'false_positive': 'int', 'false_negative': 'int', 'true_positive': 'int', 'false_positive_rate': 'float', 'false_negative_rate': 'float', 'threshold': 'float'}


In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_xgboost()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    [
        "balanced_accuracy",
        "f1",
        "recall",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).head(10)

Selected threshold: 0.4100000000000002


,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
39,0.691292,0.880068,0.538553,0.668203,0.739064,0.834281,0.719180,0.899807,8873,988,6212,7250,0.100193,0.461447,0.59
41,0.691163,0.880764,0.537736,0.667774,0.739064,0.834281,0.719177,0.900619,8881,980,6223,7239,0.099381,0.462264,0.61
42,0.691077,0.880920,0.537439,0.667589,0.739064,0.834281,0.719130,0.900821,8883,978,6227,7235,0.099179,0.462561,0.62
40,0.691163,0.880394,0.538033,0.667896,0.739064,0.834281,0.719123,0.900213,8877,984,6219,7243,0.099787,0.461967,0.60
44,0.690992,0.881263,0.536993,0.667344,0.739064,0.834281,0.719110,0.901227,8887,974,6233,7229,0.098773,0.463007,0.64
37,0.691506,0.878214,0.540484,0.669150,0.739064,0.834281,0.719081,0.897678,8852,1009,6186,7276,0.102322,0.459516,0.57
43,0.690992,0.880984,0.537216,0.667436,0.739064,0.834281,0.719069,0.900923,8884,977,6230,7232,0.099077,0.462784,0.63
35,0.691978,0.874940,0.544124,0.670972,0.739064,0.834281,0.718974,0.893824,8814,1047,6137,7325,0.106176,0.455876,0.55
38,0.691163,0.879195,0.538999,0.668294,0.739064,0.834281,0.718947,0.898895,8864,997,6206,7256,0.101105,0.461001,0.58
45,0.690734,0.881269,0.536473,0.666944,0.739064,0.834281,0.718901,0.901328,8888,973,6240,7222,0.098672,0.463527,0.65


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.6559619259957982,
 'precision': 0.6987863722766486,
 'recall': 0.7099985143366513,
 'f1': 0.7043478260869566,
 'roc_auc': 0.7390639448578895,
 'average_precision': 0.8342811689796428,
 'balanced_accuracy': 0.6460954948724125,
 'specificity': 0.5821924754081736,
 'true_negative': 5741,
 'false_positive': 4120,
 'false_negative': 3904,
 'true_positive': 9558,
 'false_positive_rate': 0.4178075245918264,
 'false_negative_rate': 0.2900014856633487,
 'threshold': 0.4100000000000002}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "xgboost"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)